# GPU-Accelerated Housing EDA with RAPIDS

This notebook uses the standardized columns from `sf_rent_cleaned.csv`. RAPIDS cuDF loads and summarizes the data on an NVIDIA GPU; pandas and Seaborn display the small result tables and charts on the CPU.

Select **Runtime → Change runtime type → T4 GPU**, then upload `sf_rent_cleaned.csv` using the Files panel. Run the cells in order. The CSV upload belongs to the runtime and may need to be repeated after a runtime reset.

Each row is a housing-unit submission. The same apartment can appear in several submission years. Rent and size estimates use the range midpoints from the original analysis. Missing or open-ended rent ranges have no usable midpoint. Exact zero rent remains included. Numeric room counts exclude `5+ Bedrooms`, `3+ Bathrooms`, shared facilities, `Others`, and missing categories.

Sources: [RAPIDS requirements](https://docs.rapids.ai/install/), [Colab setup](https://docs.nvidia.com/datascience/deployment/stable/platforms/colab/), [cuDF groupby](https://docs.rapids.ai/api/cudf/legacy/user_guide/groupby/).


In [ ]:
import subprocess
import sys
from pathlib import Path

check = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(check.stdout)
if check.returncode:
    raise RuntimeError("Select a GPU runtime before running this notebook.")

import cudf
import cupy as cp
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib.ticker import StrMethodFormatter

print("Python:", sys.version.split()[0])
print("cuDF:", cudf.__version__)
print("pandas:", pd.__version__)


## Load the cleaned CSV onto the GPU
Only the six numeric columns needed for these analyses are loaded. The saved pandas index in the CSV is ignored.


In [ ]:
csv_path = Path("/content/sf_rent_cleaned.csv")
if not csv_path.exists():
    raise FileNotFoundError("Upload sf_rent_cleaned.csv using the left Files panel, then rerun this cell.")

columns = [
    "submission_year", "bedroom_numeric", "bathroom_numeric",
    "monthly_rent_midpoint", "square_footage_midpoint", "property_age"
]
gdf = cudf.read_csv(str(csv_path), usecols=columns,
                    dtype={column: "float64" for column in columns})
features = [column for column in columns if column != "submission_year"]
print(f"Submission records loaded onto GPU: {len(gdf):,}")
display(gdf.head().to_pandas())


## Summary statistics and missing values
Compute these summaries on the GPU, then transfer only the small results to the CPU for display. Property age retains the values saved by the original notebook; it is not recalculated here.


In [ ]:
display(gdf[features].describe().to_pandas())
missing = gdf.isna().sum().to_pandas().rename("missing_records").to_frame()
missing["missing_percent"] = missing["missing_records"] / len(gdf) * 100
display(missing)


## Median estimated rent by submission year
The count is the number of usable rent midpoints, rather than the total number of submissions. Differences between years do not measure rent changes for the same apartments.


In [ ]:
yearly_rent_gpu = (
    gdf.groupby("submission_year")["monthly_rent_midpoint"]
    .agg(["median", "count"]).sort_index()
)
yearly_rent = yearly_rent_gpu.to_pandas()
yearly_rent.columns = ["median_estimated_rent", "usable_rent_records"]
display(yearly_rent)
ax = yearly_rent["median_estimated_rent"].plot(marker="o", figsize=(9, 4))
ax.set_title("Median Estimated Monthly Rent by Submission Year")
ax.set_xlabel("Submission Year")
ax.set_ylabel("Median Estimated Monthly Rent ($)")
ax.set_xticks(yearly_rent.index)
ax.set_xticklabels([f"{year:.0f}" for year in yearly_rent.index])
ax.yaxis.set_major_formatter(StrMethodFormatter("${x:,.0f}"))
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## Median rent by bedroom and bathroom count

The filtering, grouping, median, and record-count calculations run on the GPU. Bedrooms and bathrooms are shown in ascending numeric order. Each chart cell displays a median and its sample size (`n`). Combinations without usable records display `0` with `n=0` as a presentation placeholder; they do not represent an observed zero rent. Very small groups should be interpreted cautiously.


In [ ]:
room_columns = ["bedroom_numeric", "bathroom_numeric", "monthly_rent_midpoint"]

def summarize_rooms(frame):
    return (
        frame.dropna(subset=room_columns)
        .groupby(["bedroom_numeric", "bathroom_numeric"])["monthly_rent_midpoint"]
        .agg(["median", "count"]).sort_index()
    )

room_summary_gpu = summarize_rooms(gdf)
room_summary = room_summary_gpu.to_pandas()
print(f"Usable room-and-rent records: {int(room_summary['count'].sum()):,}")
display(room_summary)

# Explicitly sort both axes after converting the GPU summary to pandas.
rent_heatmap = (
    room_summary["median"].unstack("bathroom_numeric").astype(float)
    .sort_index().sort_index(axis=1)
)
record_counts = (
    room_summary["count"].unstack("bathroom_numeric")
    .reindex(index=rent_heatmap.index, columns=rent_heatmap.columns)
    .fillna(0).astype(int)
)

# Fill the display only; missing medians remain missing in rent_heatmap.
plot_rent = rent_heatmap.fillna(0)
annotations = pd.DataFrame("", index=rent_heatmap.index, columns=rent_heatmap.columns)
for bedroom in rent_heatmap.index:
    for bathroom in rent_heatmap.columns:
        rent = rent_heatmap.loc[bedroom, bathroom]
        count = record_counts.loc[bedroom, bathroom]
        annotations.loc[bedroom, bathroom] = (
            f"${rent:,.0f}\n(n={count:,})" if pd.notna(rent) else "0\n(n=0)"
        )

print("Bathroom order:", ", ".join(f"{value:g}" for value in plot_rent.columns))
fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(
    plot_rent, annot=annotations, fmt="", cmap="YlOrRd", vmin=0,
    linewidths=0.5, linecolor="white",
    cbar_kws={"label": "Median Estimated Monthly Rent ($)",
              "format": StrMethodFormatter("${x:,.0f}")}, ax=ax
)
ax.set_title("Median Estimated Monthly Rent by Bedrooms and Bathrooms\nAll Submission Years — RAPIDS cuDF Summaries")
ax.set_xlabel("Number of Bathrooms")
ax.set_ylabel("Number of Bedrooms")
ax.set_xticklabels([f"{value:g}" for value in plot_rent.columns], rotation=0)
ax.set_yticklabels(["Studio (0)" if value == 0 else f"{value:g}"
                    for value in plot_rent.index], rotation=0)
fig.text(0.5, 0.02, "0 with n=0 means no usable records; it is not an observed $0 rent.",
         ha="center", fontsize=10)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()


## Correlation between numeric features
Pearson correlation measures a straight-line association between two numeric features. Each pair uses its available values, so sample sizes can differ. Correlation does not establish causation.


In [ ]:
# This cuDF version requires null-free inputs for DataFrame.corr().
# Drop missing values separately for each pair to retain available records.
correlation = pd.DataFrame(np.nan, index=features, columns=features, dtype=float)
pair_counts = pd.DataFrame(0, index=features, columns=features, dtype=int)

for i, first_feature in enumerate(features):
    for second_feature in features[i:]:
        pair_columns = (
            [first_feature] if first_feature == second_feature
            else [first_feature, second_feature]
        )
        pair_gpu = gdf[pair_columns].dropna()
        pair_counts.loc[first_feature, second_feature] = len(pair_gpu)
        pair_counts.loc[second_feature, first_feature] = len(pair_gpu)

        if len(pair_gpu) >= 2:
            # Calculate on the GPU; transfer only this tiny result matrix.
            pair_result = pair_gpu.corr().to_pandas().astype(float)
            value = pair_result.loc[first_feature, second_feature]
            correlation.loc[first_feature, second_feature] = value
            correlation.loc[second_feature, first_feature] = value

print("Pairwise Pearson correlations calculated on the GPU:")
display(correlation)
print("Usable records for each feature pair:")
display(pair_counts)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    correlation, annot=True, fmt=".2f", cmap="RdBu_r",
    vmin=-1, vmax=1, center=0, mask=correlation.isna(), ax=ax
)
ax.set_title("Correlation of Housing Features — Calculated with RAPIDS cuDF")
plt.tight_layout()
plt.show()


## Measure CPU versus GPU aggregation time
This comparison runs the same room-summary calculation on pandas and cuDF using identical values. Both implementations are warmed up once, then measured five times. GPU synchronization makes sure the work finishes before the timer stops. Results are checked for agreement.

These timings cover in-memory filtering, grouping, median, count, and sorting only. They exclude CSV reading, uploads, initial GPU setup, transfers, and plotting; they do not measure the complete notebook. A GPU is not guaranteed to be faster for this workload. Report the measured ratio, even when it is below 1.


In [ ]:
from time import perf_counter

# Transfer outside the timed region so both implementations start in memory.
cpu_df = gdf.to_pandas()
cpu_result = summarize_rooms(cpu_df)
gpu_result = summarize_rooms(gdf)
cp.cuda.runtime.deviceSynchronize()

pd.testing.assert_frame_equal(
    cpu_result, gpu_result.to_pandas(), check_dtype=False,
    check_exact=False, rtol=1e-8, atol=1e-8
)

def measure_summary(frame, gpu=False, repeats=5):
    measurements = []
    for _ in range(repeats):
        if gpu:
            cp.cuda.runtime.deviceSynchronize()
        start = perf_counter()
        result = summarize_rooms(frame)
        if gpu:
            cp.cuda.runtime.deviceSynchronize()
        measurements.append(perf_counter() - start)
    return float(np.median(measurements))

cpu_seconds = measure_summary(cpu_df)
gpu_seconds = measure_summary(gdf, gpu=True)
timing = pd.DataFrame({
    "implementation": ["pandas (CPU)", "cuDF (GPU)"],
    "median_seconds": [cpu_seconds, gpu_seconds]
})
display(timing)
print(f"CPU time / GPU time: {cpu_seconds / gpu_seconds:.2f}x")
print("Values above 1 indicate faster GPU aggregation; values below 1 indicate faster CPU aggregation.")
print("CPU and GPU summary results agree.")
